Below is a claims pipeline:
- handles full claim narratives (up to 4k tokens)
- compares Clinical‑Longformer (long‑context) vs Bio_ClinicalBERT (short‑context)
- shows performance + practical differences

## 00 - imports and device

In [1]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

import shap
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

IPython could not be loaded!


'cuda'

## 01 - Load your dataset

In [2]:
df = pd.read_json("synthetic_claims.json")
df.head()

,id,severity,theme,department,claim_text
0,1,high,diagnostic_error,ED/Cardiology,"The patient, a 68-year-old male with hypertens..."
1,2,high,failure_to_escalate,Surgery/ICU,A 54-year-old female underwent elective laparo...
2,3,low,diagnostic_error,ED/Fracture Clinic,A 32-year-old male attended A&E after falling ...
3,4,high,delay_in_treatment,ED/Respiratory,A 76-year-old patient with COPD presented with...
4,5,high,fetal_monitoring_failure,Maternity,A 29-year-old primigravida presented in labour...


## 02 - Map severity

In [3]:
severity_map = {"low": 0, "moderate": 1, "high": 2}
df["severity_label"] = df["severity"].map(severity_map)

## 03 - train/validation split

In [4]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [5]:
train_df, val_df

(    id  severity                  theme   department  \
 55  56  moderate  communication_failure           GP   
 88  89      high         surgical_error      Theatre   
 26  27      high       diagnostic_error           ED   
 42  43      high    failure_to_escalate           ED   
 69  70       low   administrative_delay  Outpatients   
 ..  ..       ...                    ...          ...   
 60  61      high       diagnostic_error           ED   
 71  72  moderate       medication_error     Pharmacy   
 14  15      high       diagnostic_error           ED   
 92  93      high    failure_to_escalate           ED   
 51  52  moderate       medication_error     Pharmacy   
 
                                            claim_text  severity_label  
 55  A patient was not informed of abnormal liver f...               1  
 88  A surgical instrument was retained during proc...               2  
 26  A 39-year-old female presented with chest pain...               2  
 42  A patient with se

## 04 - Convert to huggingface dataset

In [6]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

## 05 Load tokenisers and models

In [7]:
BERT_NAME = "emilyalsentzer/Bio_ClinicalBERT"
LONGFORMER_NAME = "yikuan8/Clinical-Longformer"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
long_tokenizer = AutoTokenizer.from_pretrained(LONGFORMER_NAME)

bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=3).to(DEVICE)
long_model = AutoModelForSequenceClassification.from_pretrained(LONGFORMER_NAME, num_labels=3).to(DEVICE)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at yikuan8/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 06 - Tokenisation functions

In [8]:
def tokenize_bert(batch):
    return bert_tokenizer(
        batch["claim_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

def tokenize_longformer(batch):
    return long_tokenizer(
        batch["claim_text"],
        padding="max_length",
        truncation=True,
        max_length=2048
    )

## 07 - Apply tokenisation

In [9]:
bert_train = train_ds.map(tokenize_bert, batched=True)
bert_val = val_ds.map(tokenize_bert, batched=True)

long_train = train_ds.map(tokenize_longformer, batched=True)
long_val = val_ds.map(tokenize_longformer, batched=True)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 08 - Rename label column

In [10]:
bert_train = bert_train.rename_column("severity_label", "labels")
bert_val = bert_val.rename_column("severity_label", "labels")

long_train = long_train.rename_column("severity_label", "labels")
long_val = long_val.rename_column("severity_label", "labels")

## 09 - Remove unused columns

In [11]:
cols_to_remove = ["id", "severity", "theme", "department", "claim_text"]

bert_train = bert_train.remove_columns(cols_to_remove)
bert_val = bert_val.remove_columns(cols_to_remove)

long_train = long_train.remove_columns(cols_to_remove)
long_val = long_val.remove_columns(cols_to_remove)

## 10 - Set format for pytorch

In [12]:
bert_train.set_format("torch")
bert_val.set_format("torch")

long_train.set_format("torch")
long_val.set_format("torch")

## 11 - Metrics

In [13]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

## 12 - Training arguments

In [14]:
bert_args = TrainingArguments(
    output_dir="./bert-claims",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

long_args = TrainingArguments(
    output_dir="./longformer-claims",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

## 13 - Trainers

In [15]:
bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_train,
    eval_dataset=bert_val,
    compute_metrics=compute_metrics
)

long_trainer = Trainer(
    model=long_model,
    args=long_args,
    train_dataset=long_train,
    eval_dataset=long_val,
    compute_metrics=compute_metrics
)

## 14 - train and evaluation

In [16]:
bert_trainer.train()
bert_metrics = bert_trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.986100,0.870271,0.550000,0.296552
2,0.810700,0.614962,0.850000,0.577540
3,0.598300,0.392930,0.950000,0.964519
4,0.381900,0.317801,0.950000,0.964519
5,0.298200,0.281058,0.950000,0.964519


In [17]:
long_trainer.train()
long_metrics = long_trainer.evaluate()

Initializing global attention on CLS token...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.973700,0.678658,0.650000,0.413580
2,0.326400,0.312686,0.950000,0.964519
3,0.054400,0.351662,0.950000,0.964519
4,0.026800,0.567620,0.900000,0.929630
5,0.012400,0.496565,0.900000,0.929630


In [18]:
bert_metrics, long_metrics

({'eval_loss': 0.28105807304382324,
  'eval_accuracy': 0.95,
  'eval_f1_macro': 0.9645191409897292,
  'eval_runtime': 0.2748,
  'eval_samples_per_second': 72.784,
  'eval_steps_per_second': 18.196,
  'epoch': 5.0},
 {'eval_loss': 0.49656468629837036,
  'eval_accuracy': 0.9,
  'eval_f1_macro': 0.9296296296296296,
  'eval_runtime': 98.1166,
  'eval_samples_per_second': 0.204,
  'eval_steps_per_second': 0.102,
  'epoch': 5.0})

In [19]:
import pandas as pd

performance_comparison = pd.DataFrame({
    "Model": ["Bio_ClinicalBERT", "Clinical-Longformer"],
    "Context_Length": ["256 tokens", "2048 tokens"],
    "Accuracy": [bert_metrics["eval_accuracy"], long_metrics["eval_accuracy"]],
    "F1_macro": [bert_metrics["eval_f1_macro"], long_metrics["eval_f1_macro"]],
    "Notes": [
        "Short context — truncates long claims",
        "Long context — sees full narrative"
    ]
})

performance_comparison

,Model,Context_Length,Accuracy,F1_macro,Notes
0,Bio_ClinicalBERT,256 tokens,0.95,0.964519,Short context — truncates long claims
1,Clinical-Longformer,2048 tokens,0.90,0.929630,Long context — sees full narrative


In [20]:
print("=== MODEL PERFORMANCE COMPARISON ===")
print(f"BERT Accuracy:      {bert_metrics['eval_accuracy']:.3f}")
print(f"BERT F1 (macro):    {bert_metrics['eval_f1_macro']:.3f}")

print(f"Longformer Accuracy:{long_metrics['eval_accuracy']:.3f}")
print(f"Longformer F1:      {long_metrics['eval_f1_macro']:.3f}")

print("\nInterpretation:")
print("- BERT truncates at 256 tokens → loses late escalation, deterioration, ICU transfer.")
print("- Longformer processes full claim narratives → more stable severity predictions.")

=== MODEL PERFORMANCE COMPARISON ===
BERT Accuracy:      0.950
BERT F1 (macro):    0.965
Longformer Accuracy:0.900
Longformer F1:      0.930

Interpretation:
- BERT truncates at 256 tokens → loses late escalation, deterioration, ICU transfer.
- Longformer processes full claim narratives → more stable severity predictions.


## 15 - shap explainer

In [21]:
def longformer_predict_proba(texts):
    inputs = long_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=2048,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = long_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    return probs.cpu().numpy()


In [22]:
long_explainer = shap.Explainer(longformer_predict_proba, long_tokenizer)

## 16 - explain a claim

In [23]:
claim = df["claim_text"].iloc[0]
long_shap_values = long_explainer([claim])
shap.plots.text(long_shap_values[0])

In the future `np.bool` will be defined as the corresponding NumPy scalar.


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

## 17 - attention visualisation (longformer)

In [ ]:
inputs = long_tokenizer(
    claim,
    padding=True,
    truncation=True,
    max_length=2048,
    return_tensors="pt"
).to(DEVICE)

with torch.no_grad():
    outputs = long_model(**inputs, output_attentions=True)

attentions = outputs.attentions

In [ ]:
# heatmap for layer 0, head 0
layer = 0
head = 0

att = attentions[layer][0, head].cpu().numpy()

plt.figure(figsize=(10, 8))
sns.heatmap(att, cmap="viridis")
plt.title("Longformer Attention — Layer 0, Head 0")
plt.show()

## 18 - Combined SHAP + Attention

In [ ]:
tokens = long_tokenizer.tokenize(claim)
shap_vals = long_shap_values[0].values[:len(tokens)]

norm_shap = (shap_vals - shap_vals.min()) / (shap_vals.max() - shap_vals.min() + 1e-8)

plt.figure(figsize=(14, 5))
plt.bar(range(len(tokens)), norm_shap, color="red")
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.title("SHAP Token Importance (normalised)")
plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(att, cmap="viridis")
plt.title("Attention Heatmap (Layer 0, Head 0)")
plt.show()

## 19 - BERT vs Longformer SHAP comparison

In [ ]:
def bert_predict_proba(texts):
    inputs = bert_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = bert_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    return probs.cpu().numpy()

bert_explainer = shap.Explainer(bert_predict_proba, bert_tokenizer)
bert_shap_values = bert_explainer([claim])

shap.plots.text(bert_shap_values[0])
shap.plots.text(long_shap_values[0])

You’ll typically see:
- BERT truncates long narratives at 256 tokens → may miss late escalation, ICU transfer, delays, etc.
- Longformer sees the entire claim narrative → more stable severity predictions, better handling of complex timelines.

Talk about:
- truncation risk in short‑context models
- value of long‑context models for real NHS claims
- trade‑offs: speed vs context vs accuracy